# Amazon Reviews Sentiment Analysis
This notebook builds an **LSTM-based binary classifier** that decides whether
an Amazon product review is *positive* (Score > 3) or *negative/neutral* (Score ≤ 3).

### Pipeline overview
1. Load & explore the data
2. Clean null rows and engineer the label
3. Pre-process text (lower-case, strip punctuation, remove stop-words)
4. Train / test split
5. Fit tokenizer
6. Build the LSTM model with dropout for regularisation
7. Train with early stopping to prevent overfitting
8. Evaluate on the held-out test set
9. Run a quick Test example

## 1 · Imports

In [28]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

## 2 · Load & explore the data

In [29]:
df = pd.read_csv("/content/Reviews.csv")
print('Shape:', df.shape)
df.head()


Shape: (570338, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1.0,1.0,5.0,1.303862e+09,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0.0,0.0,1.0,1.346976e+09,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1.0,1.0,4.0,1.219018e+09,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3.0,3.0,2.0,1.307923e+09,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0.0,0.0,5.0,1.350778e+09,Great taffy,Great taffy at a great price. There was a wid...


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570338 entries, 0 to 570337
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Id                      570338 non-null  int64  
 1   ProductId               570338 non-null  object 
 2   UserId                  570338 non-null  object 
 3   ProfileName             570312 non-null  object 
 4   HelpfulnessNumerator    570337 non-null  float64
 5   HelpfulnessDenominator  570337 non-null  float64
 6   Score                   570337 non-null  float64
 7   Time                    570337 non-null  float64
 8   Summary                 570310 non-null  object 
 9   Text                    570337 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 43.5+ MB


In [31]:
df.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,570338.000000,570337.000000,570337.000000,570337.000000,5.703370e+05
mean,284660.583021,1.742561,2.227686,4.183171,1.296253e+09
std,164000.052030,7.626745,8.279470,1.310209,4.805616e+07
min,1.000000,0.000000,0.000000,1.000000,9.393408e+08
25%,142585.250000,0.000000,0.000000,4.000000,1.271290e+09
50%,285169.500000,0.000000,1.000000,5.000000,1.311120e+09
75%,425869.750000,2.000000,2.000000,5.000000,1.332720e+09
max,568454.000000,866.000000,923.000000,5.000000,1.351210e+09


In [32]:
# Check for missing values
df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,26
HelpfulnessNumerator,1
HelpfulnessDenominator,1
Score,1
Time,1
Summary,28
Text,1


## 3 · Label engineering
We keep only the two columns we need and convert the 1-5 star Score into a
binary label: **1 = positive** (Score > 3), **0 = negative / neutral** (Score ≤ 3).

In [36]:
# Features
df = df[['Text', 'Score']].dropna()

# Binary label: positive review = Score greater than 3
df = df.copy()
df['label'] = (df['Score'] > 3).astype(int)

print('Class distribution:')
print(df['label'].value_counts())
df.head()


Class distribution:
label
1    445248
0    125089
Name: count, dtype: int64


,Text,Score,label
0,I have bought several of the Vitality canned d...,5.0,1
1,Product arrived labeled as Jumbo Salted Peanut...,1.0,0
2,This is a confection that has been around a fe...,4.0,1
3,If you are looking for the secret ingredient i...,2.0,0
4,Great taffy at a great price. There was a wid...,5.0,1


## 4 · Text pre-processing
Each review is:
1. Lower-cased
2. Stripped of everything that is not a letter or a space
3. Collapsed to single spaces
4. Filtered to remove common English stop-words

In [37]:
def text_preprocess(text):
    text  = text.lower()
    text  = re.sub(r'[^a-z\s]', '', text)   # keep only lowercase letters + spaces
    text  = re.sub(r'\s+', ' ', text).strip() # collapse whitespace
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

df['Text'] = df['Text'].apply(text_preprocess)
df['Text'].head()

,Text
0,bought several vitality canned dog food produc...
1,product arrived labeled jumbo salted peanutsth...
2,confection around centuries light pillowy citr...
3,looking secret ingredient robitussin believe f...
4,great taffy great price wide assortment yummy ...


## 5 · Train / test split

In [38]:
X = df['Text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

print(f'Train size : {len(X_train):,}')
print(f'Test  size : {len(X_test):,}')

Train size : 456,269
Test  size : 114,068


## 6 · Tokenisation & padding
The tokenizer converts words to integer indices.

In [39]:
# Build vocabulary from training data only
tokenizer = Tokenizer(num_words = 30000)
tokenizer.fit_on_texts(X_train)

# Convert text → integer sequences, then pad to a fixed length
X_train = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen = 100, padding='post')
X_test  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen = 100, padding='post')

## 7 · Model architecture
| Layer | Purpose |
|---|---|
| `Embedding` | Maps token IDs to dense vectors |
| `Dropout(0.3)` | Randomly zeros 30 % of embedding outputs → reduces co-adaptation |
| `LSTM` | Captures sequential / contextual patterns in the text |
| `Dropout(0.3)` | Regularises the LSTM output before the classifier head |
| `Dense(sigmoid)` | Outputs a probability in (0, 1) for the positive class |

In [41]:
model = Sequential([
    Embedding(input_dim = 30000, output_dim = 128, input_length = 100),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid'),
])

model.compile(
    loss      = 'binary_crossentropy',
    optimizer = 'adam',
    metrics   = ['accuracy'],
)

## 8 · Training with early stopping
`EarlyStopping` monitors the **validation loss** and stops training as soon as
it stops improving.  `restore_best_weights=True` rolls the weights back to the
epoch with the lowest validation loss, so we never keep an overfit checkpoint.

In [42]:
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience = 2,
    restore_best_weights = True,
    verbose = 1,
)

history = model.fit(
    X_train, y_train,
    epochs = 20,
    batch_size = 128,
    validation_split = 0.1,
    callbacks = [early_stop],
    verbose = 1,
)

Epoch 1/20
3209/3209 ━━━━━━━━━━━━━━━━━━━━ 40s 12ms/step - accuracy: 0.8442 - loss: 0.3774 - val_accuracy: 0.9073 - val_loss: 0.2327
Epoch 2/20
3209/3209 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - accuracy: 0.9173 - loss: 0.2127 - val_accuracy: 0.9189 - val_loss: 0.2079
Epoch 3/20
3209/3209 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - accuracy: 0.9357 - loss: 0.1698 - val_accuracy: 0.9193 - val_loss: 0.2136
Epoch 4/20
3209/3209 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - accuracy: 0.9487 - loss: 0.1389 - val_accuracy: 0.9233 - val_loss: 0.2141
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 2.


## 9 · Evaluation on the held-out test set

In [43]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss     : {loss:.4f}')
print(f'Test accuracy : {accuracy:.4f}')

Test loss     : 0.2147
Test accuracy : 0.9152


## 10 · Test on new text

In [54]:
sequences = tokenizer.texts_to_sequences([text_preprocess("This product is great! I like it")])
test = pad_sequences(sequences, maxlen=100, padding = 'post')
pred = model.predict(test)

# Convert probability to class
label = 'Positive' if pred > 0.5 else 'Negative'
print("Predicted label:", label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Predicted label: Positive


In [55]:
sequences = tokenizer.texts_to_sequences([text_preprocess('Terrible quality, broke after one day. Very disappointed.')])
test = pad_sequences(sequences, maxlen=100, padding = 'post')
pred = model.predict(test)

# Convert probability to class
label = 'Positive' if pred > 0.5 else 'Negative'
print("Predicted label:", label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Predicted label: Negative
